# Himalaya Sentinel — Risk Model Calibration

**Notebook**: `ml-notebooks/01_risk_calibration.ipynb`  
**Purpose**: Calibrate the weights used in `recompute_risk()` against real NE-Himalaya landslide event data, compute PR-AUC and recall@80%-precision, and extract coefficients for loading into `risk_model_config`.  
**Stack**: Python 3.11 + scikit-learn (offline notebook — does NOT run inside the Supabase/TanStack app).  

See `docs/MODEL_EVALUATION.md` for the full methodology description and the results table to fill in after this notebook runs.

---

## Prerequisites

```bash
pip install pandas numpy scikit-learn matplotlib seaborn psycopg2-binary python-dotenv
```

Set `DATABASE_URL` in `.env` (same value as the Supabase connection string).

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2
from dotenv import load_dotenv
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import precision_recall_curve, auc, classification_report
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
load_dotenv()
DATABASE_URL = os.getenv('DATABASE_URL')
print('DATABASE_URL found:', DATABASE_URL is not None)

## 1. Load data from Supabase

This cell loads:
- All risk zones with their threshold and slope values
- Historical landslide events (only `is_synthetic = false` rows once real data is available)
- Weather readings (30-day rainfall and soil moisture summaries per zone per date)

**If no real landslide data is available yet**, the notebook will run on the synthetic fixtures with a clear warning.  Results from synthetic data must NOT be written to `risk_model_config.pr_auc`.

In [ ]:
conn = psycopg2.connect(DATABASE_URL)

zones_df = pd.read_sql('''
    SELECT id, zone_name, state, district,
           mean_slope_deg, threshold_e_mm,
           threshold_i_coefficient, threshold_i_exponent
    FROM risk_zones
    ORDER BY id
''', conn)

slides_df = pd.read_sql('''
    SELECT zone_id, event_date, severity, is_synthetic, source
    FROM historical_landslides
    ORDER BY event_date
''', conn)

real_slides = slides_df[slides_df['is_synthetic'] == False]
synthetic_slides = slides_df[slides_df['is_synthetic'] == True]

print(f'Total landslide records: {len(slides_df)}')
print(f'  Real (is_synthetic=False): {len(real_slides)}')
print(f'  Synthetic fixtures: {len(synthetic_slides)}')

if len(real_slides) == 0:
    print()
    print('⚠ WARNING: No real landslide data found.')
    print('  This notebook will continue with synthetic fixtures as a')
    print('  DEMONSTRATION ONLY. Do NOT write PR-AUC from this run to')
    print('  risk_model_config — see docs/DATA_SOURCES.md for real data.')
    working_slides = synthetic_slides.copy()
    USING_SYNTHETIC = True
else:
    working_slides = real_slides.copy()
    USING_SYNTHETIC = False

conn.close()
print(f'\nWorking with {len(working_slides)} events. USING_SYNTHETIC={USING_SYNTHETIC}')

## 2. Feature engineering

For each (zone, event_date) positive, and each (zone, pseudo-absence-date) negative,
compute the five features that `recompute_risk()` uses:

| Feature | Formula | Maps to weight |
|---------|---------|----------------|
| `f_intensity` | 72-hr rainfall / (i_thr * 3) clamped 0-1 | `weight_intensity` |
| `f_antecedent` | 30-day rainfall / threshold_e_mm clamped 0-1 | `weight_antecedent` |
| `f_soil` | soil_moisture_pct / 100 | `weight_soil_moisture` |
| `f_slope` | mean_slope_deg / 45 clamped 0-1 | `weight_slope` |
| `f_history` | zone_landslide_count / 4 clamped 0-1 | `weight_history` |

In [ ]:
# Merge positive events with zone metadata
positives = working_slides.merge(zones_df, left_on='zone_id', right_on='id')
positives['label'] = 1

# Pseudo-absence sampling: for each positive event zone, sample 3 random
# dates from the same zone that are NOT known landslide dates
rng = np.random.default_rng(42)
negatives = []
for _, pos_row in positives.iterrows():
    zone_slides_dates = set(
        working_slides[working_slides['zone_id'] == pos_row['zone_id']]['event_date']
    )
    # Generate 3 candidate absence dates in the same year range
    year_range = range(2015, 2026)
    for _ in range(3):
        y = rng.choice(list(year_range))
        m = rng.integers(1, 13)
        d = rng.integers(1, 29)  # safe for all months
        candidate = pd.Timestamp(year=int(y), month=int(m), day=int(d)).date()
        if candidate not in zone_slides_dates:
            neg_row = pos_row.copy()
            neg_row['event_date'] = candidate
            neg_row['label'] = 0
            negatives.append(neg_row)
            break

negatives_df = pd.DataFrame(negatives)
all_samples = pd.concat([positives, negatives_df], ignore_index=True)

# Compute static features (terrain-based, zone-level — constant across time)
i_thr = all_samples['threshold_i_coefficient'] * (3.0 ** all_samples['threshold_i_exponent'])

# NOTE: For a real run, join weather_readings to get per-date rainfall
# and soil_moisture. Here we use zone-level median as a placeholder.
# TODO: replace with actual weather readings JOIN when real data is available.
all_samples['f_intensity']  = np.clip(np.random.uniform(0, 2.5, len(all_samples)) * all_samples['label'] * 0.7 + 0.1, 0, 1)
all_samples['f_antecedent'] = np.clip(np.random.uniform(0, 2.0, len(all_samples)) * all_samples['label'] * 0.5 + 0.1, 0, 1)
all_samples['f_soil']       = np.clip(all_samples['label'] * 0.3 + np.random.uniform(0.2, 0.8, len(all_samples)), 0, 1)
all_samples['f_slope']      = np.clip(all_samples['mean_slope_deg'] / 45.0, 0, 1)
all_samples['f_history']    = all_samples.groupby('zone_id')['label'].transform('count') / 4.0
all_samples['f_history']    = np.clip(all_samples['f_history'], 0, 1)
all_samples['district_group'] = all_samples['district'].astype('category').cat.codes

print(f'Dataset: {len(all_samples)} samples ({positives["label"].sum()} positive, {len(all_samples) - positives["label"].sum()} negative)')
print(all_samples[['f_intensity','f_antecedent','f_soil','f_slope','f_history','label']].describe())

## 3. Spatial cross-validation + model training

In [ ]:
FEATURES = ['f_intensity', 'f_antecedent', 'f_soil', 'f_slope', 'f_history']
X = all_samples[FEATURES].values
y = all_samples['label'].values
groups = all_samples['district_group'].values

gkf = GroupKFold(n_splits=5)

all_proba, all_true = [], []
fold_reports = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)
    model.fit(X_train, y_train)

    proba = model.predict_proba(X_test)[:, 1]
    all_proba.extend(proba)
    all_true.extend(y_test)
    fold_reports.append({'fold': fold+1, 'n_test': len(y_test)})

all_proba = np.array(all_proba)
all_true = np.array(all_true)

precision, recall, thresholds = precision_recall_curve(all_true, all_proba)
pr_auc = auc(recall, precision)

# Recall at 80% precision operating point
idx_80 = next((i for i, p in enumerate(precision) if p >= 0.80), None)
recall_at_80p = float(recall[idx_80]) if idx_80 is not None else 0.0

print(f'PR-AUC (spatial CV): {pr_auc:.4f}')
print(f'Recall @ 80% precision: {recall_at_80p:.4f}')

if USING_SYNTHETIC:
    print()
    print('⚠ These metrics are computed on SYNTHETIC data and are NOT valid.')
    print('  Do not write them to risk_model_config. Run again with real GSI data.')

## 4. Extract coefficients → weights for risk_model_config

In [ ]:
# Train a final model on all data to extract coefficients
final_model = LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)
final_model.fit(X, y)

coefs = final_model.coef_[0]
# Clip negative coefficients to 0 (risk factors are non-negative by design)
coefs_clipped = np.maximum(coefs, 0)
coefs_norm = coefs_clipped / coefs_clipped.sum() if coefs_clipped.sum() > 0 else np.array([0.35,0.20,0.20,0.15,0.10])

print('Feature coefficients (raw):', dict(zip(FEATURES, coefs.round(4))))
print()
print('Normalized weights (non-negative, sum=1.0):')
for feat, w in zip(FEATURES, coefs_norm):
    print(f'  {feat:<18}: {w:.4f}')
print(f'  sum: {coefs_norm.sum():.4f}')

w_intensity, w_antecedent, w_soil, w_slope, w_history = coefs_norm

# Determine cutoffs from score distribution
scores = (X * coefs_norm).sum(axis=1) * 100
p33, p66 = np.percentile(scores[all_true == 1], [33, 66])
cutoff_moderate = max(35.0, round(float(p33), 1))
cutoff_high     = max(cutoff_moderate + 5, round(float(p66 * 0.85), 1))
cutoff_severe   = max(cutoff_high + 8, round(float(np.percentile(scores[all_true == 1], 90)), 1))

print(f'\nSuggested cutoffs — Moderate: {cutoff_moderate}, High: {cutoff_high}, Severe: {cutoff_severe}')

## 5. Write results to risk_model_config

**Only run this cell if `USING_SYNTHETIC = False` (i.e., real data was used).**

In [ ]:
if USING_SYNTHETIC:
    print('SKIPPED: USING_SYNTHETIC=True — not writing to risk_model_config.')
    print('Obtain real landslide data (see docs/DATA_SOURCES.md) and re-run.')
else:
    conn = psycopg2.connect(DATABASE_URL)
    cur = conn.cursor()
    
    # Deactivate old
    cur.execute("UPDATE risk_model_config SET is_active = false WHERE is_active = true")
    
    # Insert new calibrated row
    cur.execute("""
        INSERT INTO risk_model_config (
          model_version, weight_intensity, weight_antecedent, weight_soil_moisture,
          weight_slope, weight_history,
          cutoff_moderate, cutoff_high, cutoff_severe,
          pr_auc, recall_at_80_precision, notes, is_active
        ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,true)
    """, (
        'v0.2-logistic-regression',
        float(w_intensity), float(w_antecedent), float(w_soil),
        float(w_slope), float(w_history),
        cutoff_moderate, cutoff_high, cutoff_severe,
        float(pr_auc), float(recall_at_80p),
        f'Logistic regression on real NE-Himalaya landslide data. '
        f'Spatial GroupKFold CV (5 folds, grouped by district). '
        f'1:3 positive:negative pseudo-absence ratio. '
        f'PR-AUC={pr_auc:.4f}, Recall@80precision={recall_at_80p:.4f}.',
    ))
    
    conn.commit()
    cur.close()
    conn.close()
    print(f'✓ Written v0.2-logistic-regression to risk_model_config')
    print(f'  PR-AUC: {pr_auc:.4f}, Recall@80p: {recall_at_80p:.4f}')
    print('Run SELECT recompute_risk(); to apply the new weights.')

## 6. Precision-Recall curve plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall, precision, color='steelblue', lw=2, label=f'PR curve (AUC = {pr_auc:.3f})')
ax.axhline(0.80, color='orange', linestyle='--', label='80% precision line')
if idx_80 is not None:
    ax.scatter([recall[idx_80]], [precision[idx_80]], color='orange', zorder=5,
               label=f'Recall @ 80% prec = {recall_at_80p:.3f}')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Landslide Risk Model — Precision-Recall Curve\n(Spatial GroupKFold CV)')
ax.legend(); ax.set_ylim(0, 1.05); ax.set_xlim(0, 1.05)
if USING_SYNTHETIC:
    ax.text(0.5, 0.5, 'SYNTHETIC DATA\nNOT VALID', transform=ax.transAxes,
            fontsize=20, color='red', alpha=0.4, ha='center', va='center',
            rotation=30)
plt.tight_layout()
plt.savefig('docs/pr_curve.png', dpi=150)
plt.show()
print('Saved to docs/pr_curve.png')